### Notebook 01 — Data exploration and understanding

Notebook 01 answers the question: What is in our dataset, and what patterns does it contain?

In [ ]:
# Import necessary libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Plot styling
sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

PROJECT_ROOT = Path.cwd()

# If Jupyter starts inside notebooks/, go up one level.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

INTERIM_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_ROOT)


### Load the core dataset

In [ ]:
core_file = RAW_PATH / "MPD_sample_synthetic_ken_3000subs.csv"

if not core_file.exists():
    raise FileNotFoundError(
        f"Dataset not found: {core_file}."
        "Check your downloaded filename and data/raw directory."
    )

df = pd.read_csv(
    core_file,
    dtype={
        "msisdn": "string",
        "cell_id": "string",
        "latitude": "float64",
        "longitude": "float64",
        "data_type": "category",
        "service": "category"
    },
    parse_dates=["datetime"]
)
print("Dataset Shape:", df.shape)
display(df.head())

### Inspect columns and data types

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("data_type"))

print("\nDataset Information:")
df.info()

print("\nUnique subscribers:")
print(df["msisdn"].nunique())

print("\nUnique cell towers:")
print(df["cell_id"].nunique())

### Missing values

In [ ]:

missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": df.isnull().mean() * 100
})

missing = missing.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing)


### Basic dataset statistics

In [ ]:
print("Total observations:", len(df))
print("Unique subscribers", df["msisdn"].nunique())
print("Unique Cell Towers:", df["cell_id"].nunique())
print("Average events per subscribers:", len(df) / df["msisdn"].nunique())
print("average events per cell:", len(df) / df["cell_id"].nunique())

### Check Timestamps

In [ ]:
print("Earliest timestamp:", df["datetime"].min())
print("Latest timestamp:", df["datetime"].max())
print("Invalid timestamps:", df["datetime"].isna().sum())


### Events by hour

In [ ]:
df["hour"] = df["datetime"].dt.hour

hourly_events = (
    df.groupby("hour")
    .size()
    .reset_index(name="event_count")
)

plt.figure(figsize=(12, 5))

sns.barplot(
    data=hourly_events,
    x="hour",
    y="event_count"
)

plt.title("Mobile Network Events by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Events")
plt.tight_layout()
plt.show()

#### Section 1 — Statistical profiling and data structure
We want to understand the distribution of every field, not just its data type.

In [ ]:
# Descriptive statistics
display(df.describe(include="all").T)

### Cardinality analysis

Cardinality tells us how many distinct values exist in each column.

In [ ]:
# Cardinality tells us how many distinct values exist in each column.

cardinality = pd.DataFrame({
    "column": df.columns,
    "unique_values": [
        df[col].nunique(dropna=False)
        for col in df.columns
    ],
    "unique_percentage": [
        df[col].nunique(dropna=False) / len(df) * 100
        for col in df.columns
    ]
})

display(
    cardinality.sort_values(
        "unique_values",
        ascending=False
    )
)

### Frequency distributions

In [ ]:
for col in ["data_type", "service"]:

    print(f"\nDistribution of {col}")

    distribution = (
        df[col]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    distribution["percentage"] = (
        distribution["count"] / len(df) * 100
    )

    display(distribution)

### Visualise network service and data types

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(
    data=df,
    x="data_type",
    order=df["data_type"].value_counts().index,
    ax=axes[0]
)

axes[0].set_title("Distribution of Data Types")
axes[0].set_xlabel("Data Type")
axes[0].set_ylabel("Number of Records")

sns.countplot(
    data=df,
    x="service",
    order=df["service"].value_counts().index,
    ax=axes[1]
)

axes[1].set_title("Distribution of Network Services")
axes[1].set_xlabel("Network Technology")
axes[1].set_ylabel("Number of Records")

plt.tight_layout()
plt.show()

### Section 2 - Subscriber activity analysis

##### Events per subscriber


### Subscriber Activity Metrics

The following metrics are calculated for each subscriber to understand their network activity, mobility patterns, and observation coverage.

| Metric | What it measures |
|:---|:---|
| `total_events` | Total number of network records generated by a subscriber. |
| `unique_cells` | Number of distinct cell towers observed for that subscriber. |
| `active_days` | Number of different days on which the subscriber generated an event. |
| `first_seen` | Timestamp of the subscriber's first recorded event. |
| `last_seen` | Timestamp of the subscriber's last recorded event. |


In [ ]:
subscriber_activity = (
    df.groupby("msisdn")
       .agg(
           total_events=("datetime", "size"),
           unique_cells=("cell_id", "nunique"),
           active_days=("datetime", lambda x: x.dt.date.nunique()),
           first_seen=("datetime", "min"),
           last_seen=("datetime", "max")
       )
       .reset_index()
)

display(subscriber_activity.describe().T)

Key findings:

1. Network activity: Subscribers generated an average of approximately 358 events, with a median of 265. The difference between the mean and median suggests a positively skewed distribution, indicating that some subscribers generate substantially more network events than others.

2. Cell tower diversity: Each subscriber interacted with an average of 111 unique cell towers, with a median of 95. This indicates considerable variation in observed geographical diversity across subscribers.

3. Observation coverage: Subscribers were active for an average of approximately 26 days, while the median was 28 days. This suggests that most subscribers have relatively consistent observation coverage throughout the study period, although some have limited observations.

4. First observation: Most subscribers were first observed on February 1, although some appeared much later in the observation period, with the latest first observation occurring on February 27.

5. Last observation: Most subscribers had their final recorded events on February 28, suggesting that a substantial proportion remained observable throughout the study period. However, some subscribers had their last recorded events much earlier.

Subscribers with more active days and higher event volumes may provide stronger observational evidence for identifying recurring geographical patterns. However, event frequency and the number of cell towers visited should not be interpreted as direct measures of physical mobility.

### Distribution of subscriber event counts

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    data=subscriber_activity,
    x="total_events",
    bins=50,
    kde=True
)

plt.title("Distribution of Events per Subscriber")
plt.xlabel("Total Events")
plt.ylabel("Number of Subscribers")

plt.show()


### Interpretation and Key Findings

The distribution of subscriber event counts exhibits a positively skewed (right-skewed) pattern, indicating considerable variation in network activity across the 3,000 subscribers.

Most subscribers generate relatively moderate numbers of network events, with the majority recording fewer than approximately 1,000 events during the 28-day observation period. However, a small number of subscribers exhibit substantially higher activity levels, resulting in a long right tail.

The descriptive statistics further support this observation, with an average of 357.54 events compared to a median of 265 events. The maximum event count of 7,527 indicates the presence of an extreme high-activity subscriber.

**Key findings:**

1. Subscriber network activity is unevenly distributed, with considerable variation in event generation.
2. The distribution is positively skewed, suggesting that a relatively small number of highly active subscribers influence the overall average.
3. Extreme observations may affect statistical summaries and should be investigated further.
4. The median provides a more representative measure of typical subscriber activity than the mean in this distribution.

**Implications for Mobility Intelligence:**

Differences in subscriber event frequency may affect the reliability of subsequent mobility analysis and residential location inference. Subscribers with higher event volumes may provide more observational evidence for identifying recurring geographical patterns, whereas subscribers with limited activity may require more cautious interpretation.

Further investigation of activity outliers, event frequency and observation coverage is recommended before proceeding with subscriber-level mobility profiling.


### Identify the most active subscribers

In [ ]:
top_subscribers = (
    subscriber_activity
    .sort_values("total_events", ascending=False)
    .head(20)
)

display(top_subscribers)

### Identify the least active subscribers

In [ ]:
least_active_subscribers = (
    subscriber_activity
    .sort_values("total_events", ascending=True)
    .head(20)
)

display(least_active_subscribers)

### Subscriber activity concentration

This helps us understand how much of the dataset is generated by the most active subscribers.

In [ ]:
activity_sorted = (
    subscriber_activity["total_events"]
    .sort_values(ascending=False)
    .reset_index(drop=True)
)

cumulative_activity = activity_sorted.cumsum() / activity_sorted.sum()

subscriber_percentile = (
    np.arange(1, len(activity_sorted) + 1)
    / len(activity_sorted)
)

plt.figure(figsize=(10, 6))

plt.plot(
    subscriber_percentile * 100,
    cumulative_activity * 100
)

plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Equal activity reference"
)

plt.title("Cumulative Share of Network Events by Subscriber")
plt.xlabel("Share of Subscribers (ranked most active first)")
plt.ylabel("Cumulative Share of Events (%)")
plt.legend()

plt.show()


### Interpretation and Key Findings

The cumulative subscriber activity analysis reveals a considerable concentration of network events among the most active subscribers.

The cumulative activity curve rises substantially above the equal activity reference, indicating that network event generation is unevenly distributed across the subscriber population.

**Key findings:**

1. The most active 20% of subscribers generate approximately 51% of total network events.
2. The top 40% of subscribers account for approximately 75% of network activity.
3. The top 60% contribute approximately 90% of all recorded events.
4. The remaining subscribers contribute a relatively small proportion of total network events.
5. The observed concentration reinforces the positively skewed distribution identified in the subscriber event-count analysis.

**Implications for Mobility Intelligence:**

The unequal distribution of network events has important implications for subscriber-level mobility analysis and residential location inference.

Highly active subscribers provide more network observations, potentially offering stronger evidence for identifying recurring geographical patterns. Conversely, subscribers generating relatively few events may have insufficient observations for reliable mobility characterization.

This suggests that subsequent analyses should account for differences in subscriber activity levels and observation coverage rather than treating all subscribers as equally observable.

**Overall conclusion:**

Network activity is disproportionately concentrated among a relatively small proportion of subscribers. Understanding this imbalance is important when assessing observation reliability, interpreting mobility patterns and developing robust residential location inference methodologies.


#### Activity versus geographic diversity

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=subscriber_activity,
    x="total_events",
    y="unique_cells",
    alpha=0.6
)

plt.title("Subscriber Activity vs Geographic Cell Diversity")
plt.xlabel("Total Network Events Generated by Subscribers")
plt.ylabel("Number of Unique Cells")

plt.show()

print(
    "Correlation:",
    subscriber_activity[
        ["total_events", "unique_cells"]
    ].corr().iloc[0, 1]
)


### Interpretation and Key Findings

The scatter plot illustrates the relationship between subscriber network activity and geographical cell diversity across the 3,000 subscribers.

The Pearson correlation coefficient of 0.4145 indicates a moderate positive relationship between the total number of network events generated by subscribers and the number of unique cell towers they interact with.

**Key findings:**

1. There is a moderate positive association between subscriber event volume and cell tower diversity. Subscribers generating more network events generally tend to interact with more unique cell towers.

2. Considerable variation exists among subscribers with similar event volumes, suggesting that network activity alone does not fully explain geographical diversity.

3. The relationship is not particularly strong, indicating that other factors, including individual mobility behaviour and observation coverage, influence the number of cell towers visited.

4. A small number of unusual observations have approximately 7,000–7,500 events but zero unique cell towers. These records require further investigation to establish whether they result from data aggregation or data quality issues.

**Implications for Mobility Intelligence:**

The positive relationship suggests that event frequency provides some information about subscriber geographical diversity. However, event volume should not be used as a direct proxy for physical mobility.

Subsequent analyses should incorporate additional spatial metrics, including radius of gyration, geographical dispersion, cell tower transition frequency and location entropy, to better understand individual subscriber mobility patterns.

**Overall conclusion:**

Subscriber network activity and geographical cell diversity exhibit a moderate positive relationship. However, the variation between subscribers demonstrates the need to investigate spatial and temporal characteristics independently of event frequency when developing reliable mobility intelligence and residential location inference methodologies.


### Section 3 — Temporal analysis

We will now investigate when network events occur.

This will help us understand whether activity is concentrated during working hours, evenings, nights or weekends.

#### Derive temporary EDA features

These are analysis features only. We are not yet performing the formal preprocessing step.

In [ ]:
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day

df["day_of_week"] = df["datetime"].dt.dayofweek
df["day_name"] = df["datetime"].dt.day_name()

df["is_weekend"] = df["day_of_week"].isin([5, 6])

df["is_night"] = (
    (df["hour"] >= 20) |
    (df["hour"] < 6)
)

#### Hourly activity

In [ ]:
hourly_activity = (
    df.groupby("hour")
       .size()
       .reindex(range(24), fill_value=0)
)

plt.figure(figsize=(14, 6))

sns.lineplot(
    x=hourly_activity.index,
    y=hourly_activity.values,
    marker="o"
)

plt.xticks(range(24))
plt.title("Hourly Mobile Network Activity")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Events")

plt.show()


### Interpretation and Key Findings

The hourly mobile network activity analysis reveals a distinct daily activity pattern across the 24 hours of the day.

Network activity is lowest during the early morning hours (02:00–04:00), followed by a sharp increase between 05:00 and 08:00. Activity reaches its highest levels between 09:00 and 10:00, with approximately 67,000 recorded events per hour.

Network activity remains relatively stable throughout the afternoon (11:00–16:00), before gradually declining during the evening. A particularly sharp reduction is observed between 20:00 and 22:00, indicating a substantial decrease in network event generation towards the end of the day.

**Key findings:**

1. Network activity is lowest between 02:00 and 04:00, suggesting reduced overnight activity.
2. A sharp increase in network events occurs between 05:00 and 08:00, potentially reflecting the beginning of daily activities and commuting patterns.
3. Peak network activity occurs between 09:00 and 10:00, with approximately 67,000 events per hour.
4. Activity remains consistently high throughout the afternoon, indicating sustained daytime network usage.
5. Network activity gradually declines during the evening, with a significant reduction after 20:00.

**Implications for Mobility Intelligence:**

The observed daily activity cycle provides useful temporal information for investigating subscriber mobility patterns.

The early morning hours may provide useful observations for identifying recurring nighttime locations, while morning and daytime observations can help investigate movement towards workplaces and other frequently visited destinations.

Separating subscriber observations into nighttime, morning, daytime and evening periods will therefore be important for subsequent spatiotemporal analysis and residential location inference.

**Overall conclusion:**

Mobile network activity exhibits a clear daily cycle, characterised by low overnight activity, a sharp morning increase, sustained daytime activity and declining evening activity. These temporal patterns provide a useful foundation for investigating recurring subscriber locations and daily mobility behaviour.


#### Daily event volume

In [ ]:
daily_activity = (
    df.groupby("date")
       .size()
       .rename("total_events")
       .reset_index()
)

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=daily_activity,
    x="date",
    y="total_events",
    marker="o"
)

plt.xticks(rotation=45)
plt.title("Daily Mobile Network Activity")
plt.xlabel("Date")
plt.ylabel("Number of Events")

plt.tight_layout()
plt.show()


### Interpretation and Key Findings

The daily mobile network activity analysis reveals relatively stable network event generation throughout the 28-day observation period.

The dataset contains 1,072,608 network events, averaging approximately 38,307 events per day. Daily event volumes generally fluctuate within a relatively narrow range of approximately 37,800 to 38,900 events.

**Key findings:**

1. Daily network activity remains relatively consistent throughout February, with most days recording approximately 38,000–38,500 events.
2. The highest daily activity occurs around February 19, with approximately 38,840 recorded events.
3. February 1 records the lowest activity, with approximately 37,820 events.
4. Noticeable daily fluctuations occur throughout the observation period, although these do not indicate a sustained upward or downward trend.
5. Overall, aggregate network activity appears relatively stable across the observation window.

**Implications for Mobility Intelligence:**

The relatively stable daily event volume suggests that the dataset provides consistent aggregate network observations throughout the study period.

However, stable daily event counts do not necessarily imply that individual subscribers exhibit consistent mobility patterns. Different subscribers may become active or inactive on different days while maintaining similar aggregate network volumes.

Further investigation should examine subscriber-level daily activity, active subscriber counts and weekday-versus-weekend variations to understand the underlying behavioural patterns.

**Overall conclusion:**

Daily network activity remains relatively stable throughout February 2026, with moderate fluctuations and no obvious sustained temporal trend. This provides a useful foundation for investigating individual subscriber activity consistency and temporal mobility patterns.


#### Day-of-week activity

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_activity = (
    df.groupby("day_name")
       .size()
       .reindex(weekday_order)
)

plt.figure(figsize=(12, 6))

sns.barplot(
    x=weekday_activity.index,
    y=weekday_activity.values
)

plt.xticks(rotation=30)
plt.title("Network Activity by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Number of Events")

plt.show()

#### Weekday versus weekend activity

In [ ]:
weekend_comparison = (
    df.groupby("is_weekend")
       .agg(
           total_events=("msisdn", "size"),
           unique_subscribers=("msisdn", "nunique"),
           unique_cells=("cell_id", "nunique")
       )
)

weekend_comparison["events_per_subscriber"] = (
    weekend_comparison["total_events"] /
    weekend_comparison["unique_subscribers"]
)

display(weekend_comparison)


### Interpretation and Key Findings

The weekday-versus-weekend analysis reveals considerably higher mobile network activity during weekdays.

Weekdays account for approximately 71.5% of total network events, compared with 28.5% during weekends. Average events per observed subscriber are approximately 256 on weekdays and 104 on weekends.

Despite this difference in event volume, subscriber coverage remains relatively similar across both periods, suggesting that the difference is primarily associated with network activity intensity rather than subscriber participation alone.

Weekdays also exhibit slightly greater geographical cell diversity than weekends.

**Overall conclusion:**

Subscribers generate substantially more network events during weekdays than weekends, potentially reflecting differences in working routines, commuting and daily activities. These differences provide useful context for subsequent temporal mobility and residential location inference analyses.


#### Hour versus day heatmap

In [ ]:
hour_day_matrix = pd.crosstab(
    df["day_name"],
    df["hour"]
)

hour_day_matrix = hour_day_matrix.reindex(weekday_order)

plt.figure(figsize=(16, 7))

sns.heatmap(
    hour_day_matrix,
    cmap="YlOrRd",
    linewidths=0.2
)

plt.title("Network Activity by Day and Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")

plt.show()


### Interpretation and Key Findings

The hour-versus-day heatmap reveals a consistent daily network activity pattern across all seven days of the week.

**Key findings:**

1. Network activity is lowest between 02:00 and 05:00, indicating reduced overnight activity.
2. A sharp increase occurs between 06:00 and 08:00, potentially reflecting morning routines and commuting.
3. Peak network activity occurs between 09:00 and 17:00, with sustained high activity throughout the daytime.
4. Activity gradually declines during the evening, particularly after 20:00.
5. Similar hourly activity patterns are observed across weekdays and weekends, suggesting relatively consistent daily network usage patterns.

**Implications for Mobility Intelligence:**

The consistency of hourly activity patterns provides a useful foundation for defining temporal windows for subscriber mobility analysis.

Nighttime observations can support residential location inference, while morning and daytime observations can help investigate commuting behaviour and frequently visited locations.

**Overall conclusion:**

Mobile network activity follows a relatively consistent daily cycle across the week, characterised by low overnight activity, a sharp morning increase, sustained daytime activity and declining evening activity.


#### Daily active subscribers

In [ ]:
daily_active_subscribers = (
    df.groupby("date")["msisdn"]
       .nunique()
       .rename("active_subscribers")
       .reset_index()
)

plt.figure(figsize=(14, 5))

sns.lineplot(
    data=daily_active_subscribers,
    x="date",
    y="active_subscribers",
    marker="o"
)

plt.title("Daily Active Subscribers")
plt.xlabel("Date")
plt.ylabel("Unique Active Subscribers")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Interpretation and Key Findings

The daily active subscriber analysis reveals relatively consistent subscriber participation throughout February 2026.

Daily active subscribers generally range between approximately 2,760 and 2,795, out of the 3,000 unique subscribers in the dataset.

**Key findings:**

1. Most subscribers remain active throughout the observation period, with daily participation generally exceeding 90% of the total subscriber population.
2. February 15 records the highest daily participation, with approximately 2,794 active subscribers.
3. February 1 records the lowest participation, with approximately 2,735 active subscribers.
4. Daily subscriber participation exhibits minor fluctuations without a sustained upward or downward trend.

**Implications for Mobility Intelligence:**

The relatively high and stable daily subscriber participation suggests good temporal observation coverage across the study period.

However, consistent daily participation does not necessarily mean that the same subscribers are active every day. Subscriber-level activity consistency should be investigated to distinguish regular subscribers from temporary visitors and intermittent users.

**Overall conclusion:**

The dataset demonstrates relatively stable daily subscriber participation, providing a useful foundation for subsequent subscriber-level temporal and mobility analysis.


### Section 4 — Geographic analysis

Investigate where the network events are recorded.

The latitude and longitude represent the associated cell location, not necessarily the precise location of the subscriber's handset. A CDR/IPDR event is associated with a serving cell, which is a proxy for location.

#### Geographic summary

In [ ]:
print("Latitude statistics")
display(df["latitude"].describe())

print("\nLongitude statistics")
display(df["longitude"].describe())

print("\nUnique coordinate pairs")
print(
    df[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

#### Coordinate consistency per cell

In [ ]:
cell_geo_profile = (
    df.groupby("cell_id")
       .agg(
           latitude_values=("latitude", "nunique"),
           longitude_values=("longitude", "nunique"),
           latitude=("latitude", "median"),
           longitude=("longitude", "median"),
           events=("msisdn", "size"),
           subscribers=("msisdn", "nunique")
       )
       .reset_index()
)

inconsistent_cell_ids = cell_geo_profile[
    (cell_geo_profile["latitude_values"] > 1) |
    (cell_geo_profile["longitude_values"] > 1)
]

print("Total cell IDs:", len(cell_geo_profile))
print("Cell IDs with inconsistent coordinates:", len(inconsistent_cell_ids))

display(inconsistent_cell_ids.head(20))


### Interpretation and Key Findings

The coordinate consistency analysis reveals that 11,026 out of 35,956 unique cell IDs (approximately 30.67%) have multiple recorded latitude or longitude values.

The remaining 24,930 cell IDs (69.33%) have consistent coordinates based on the uniqueness checks performed.

**Key findings:**

1. Approximately 30.67% of cell IDs exhibit coordinate variations.
2. The remaining 69.33% have consistent geographical coordinates.
3. Coordinate variations occur across cell IDs with different event volumes and subscriber counts.
4. Median latitude and longitude can provide representative coordinates for further investigation.

**Implications for Mobility Intelligence:**

Coordinate inconsistencies could introduce inaccuracies into geographical analysis, including distance calculations, cell tower transitions, radius of gyration and residential location inference.

Further investigation is required to establish whether these variations represent genuine coordinate inconsistencies, rounding differences or characteristics of the synthetic dataset.

**Overall conclusion:**

A substantial proportion of cell IDs exhibit coordinate variations. These records should undergo further geographical validation before being used for spatial analysis and subscriber mobility modelling.


#### Plot the geographic distribution

In [ ]:
plt.figure(figsize=(12, 9))

plt.scatter(
    df["longitude"],
    df["latitude"],
    s=1,
    alpha=0.15
)

plt.title("Geographic Distribution of Network Events")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()


### Interpretation and Key Findings

The geographical distribution analysis reveals a concentration of mobile network events around central Nairobi, with additional observations extending into surrounding geographical areas.

**Key findings:**

1. Network events are heavily concentrated around longitude 36.80–36.93 and latitude -1.25 to -1.34.
2. The spatial distribution is uneven, indicating varying levels of network activity across the study area.
3. The dataset covers a relatively broad geographical area, extending beyond the central concentration.
4. Several linear and branching spatial patterns are visible, potentially reflecting geographical infrastructure and network coverage.
5. Some isolated observations require further investigation for potential coordinate anomalies.

**Implications for Mobility Intelligence:**

The geographical concentration of network events provides an initial understanding of spatial activity distribution across Nairobi.

Further spatial analysis should investigate subscriber-level geographical dispersion, recurring locations, cell tower density and movement between geographical areas.

**Overall conclusion:**

Mobile network activity is geographically concentrated around central Nairobi, with varying levels of activity across the wider study area. This provides a foundation for subsequent spatial analysis and residential location inference.


#### Plot unique cell locations

In [ ]:
unique_cells = cell_geo_profile[
    ["cell_id", "latitude", "longitude"]
].drop_duplicates("cell_id")

plt.figure(figsize=(12, 9))

plt.scatter(
    unique_cells["longitude"],
    unique_cells["latitude"],
    s=5,
    alpha=0.5
)

plt.title("Geographic Distribution of Unique Cell IDs")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()


### Interpretation and Key Findings

The geographical distribution of unique cell locations reveals a concentration of mobile network infrastructure around central Nairobi, with additional coverage extending into surrounding areas.

**Key findings:**

- Cell locations are heavily concentrated around central Nairobi.
- Network infrastructure is unevenly distributed across the study area.
- Distinct linear and branching spatial patterns are visible.
- Cell tower density varies across different geographical locations.
- Some areas exhibit relatively sparse network infrastructure coverage.

**Implications for Mobility Intelligence:**

The spatial distribution of cell towers is important for understanding network coverage and interpreting subscriber location observations.

Areas with higher cell tower density may provide greater spatial resolution for mobility analysis, while areas with fewer towers may have less precise location estimates.

**Overall conclusion:**

The dataset reveals substantial geographical variation in mobile network infrastructure across Nairobi. This provides an important foundation for subsequent spatial aggregation, subscriber mobility analysis and residential location inference.


#### Cell-level event concentration

In [ ]:
top_20_cells = (
    cell_geo_profile
    .sort_values("events", ascending=False)
    .head(20)
)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_20_cells,
    x="events",
    y="cell_id"
)

plt.title("Top 20 Cell IDs by Event Volume")
plt.xlabel("Number of Events")
plt.ylabel("Cell ID")

plt.show()

### Section 5 — Subscriber mobility and geographic diversity

This section explores how subscribers appear across different cell IDs.

This section is particularly relevant to the eventual home-location inference model.

##### Subscriber geographic diversity

In [ ]:
subscriber_geography = (
    df.groupby("msisdn")
       .agg(
           unique_cells=("cell_id", "nunique"),
           unique_coordinates=(
               "cell_id",
               lambda x: df.loc[
                   x.index,
                   ["latitude", "longitude"]
               ].drop_duplicates().shape[0]
           )
       )
       .reset_index()
)

display(subscriber_geography.describe().T)

#### Distribution of unique cells per subscriber

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    data=subscriber_geography,
    x="unique_cells",
    bins=40,
    kde=True
)

plt.title("Distribution of Unique Cell IDs per Subscriber")
plt.xlabel("Unique Cell IDs")
plt.ylabel("Number of Subscribers")

plt.show()

### Interpretation: Distribution of Unique Cells per Subscriber

The distribution of unique cell towers per subscriber is **right-skewed**, indicating substantial variation in subscribers' geographic activity.

Key observations:

1. **High concentration at lower cell counts:** A large proportion of subscribers connect to relatively few unique cell towers, particularly around 10–40 cells. This suggests that many subscribers have relatively localized movement patterns.

2. **Secondary concentration around 80–120 cells:** There is another noticeable peak, suggesting a group of subscribers with broader geographic activity.

3. **Long right tail:** Some subscribers connect to more than 200–350 unique cell towers, indicating substantially wider geographic movement.

4. **Heterogeneous mobility patterns:** The distribution suggests that subscribers exhibit different levels of geographic mobility rather than following a uniform movement pattern.

### Summary of Findings

The dataset exhibits considerable variation in subscriber mobility. While many subscribers interact with a relatively small geographic footprint, others connect to hundreds of distinct cell towers.

**Implication for the project:** Unique cell count is a useful mobility indicator, but it should not independently determine residence. We should combine it with temporal patterns, nighttime activity, repeated cell visits and geographic consistency to infer subscribers' probable residential locations.

#### Subscriber daily cell diversity

In [ ]:
daily_cell_diversity = (
    df.groupby(["msisdn", "date"])
       .agg(
           daily_unique_cells=("cell_id", "nunique"),
           daily_events=("cell_id", "size")
       )
       .reset_index()
)

display(daily_cell_diversity.describe().T)

### Interpretation: Subscriber Daily Cell Diversity

The results show how many distinct cell towers subscribers connect to per day and the volume of network events generated.

Key observations:

1. **Daily geographic diversity:** Subscribers connect to an average of approximately 6 unique cell towers per day. The median is 5, indicating that typical daily movement is relatively localized.

2. **Variation in daily movement:** The middle 50% of subscriber-days involve between 3 and 8 unique cell towers, suggesting differences in daily mobility patterns.

3. **Daily network activity:** Subscribers generate an average of approximately 14 events per day, with a median of 10 events.

4. **High-activity outliers:** Some subscriber-days record as many as 38 unique cells and 289 network events, suggesting unusually high activity or extensive movement.

### Summary of Findings

Most subscribers exhibit relatively localized daily movement, typically connecting to around 3–8 unique cell towers per day. However, a smaller proportion demonstrate substantially higher geographic diversity and network activity.

**Implication for the project:** Daily cell diversity is a useful feature for distinguishing localized subscribers from highly mobile subscribers. Combined with nighttime activity, repeated cell visits and temporal consistency, it can help improve home-location inference.

#### Distribution of daily cell diversity

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    data=daily_cell_diversity,
    x="daily_unique_cells",
    bins=30
)

plt.title("Daily Geographic Cell Diversity per Subscriber")
plt.xlabel("Unique Cell IDs per Subscriber per Day")
plt.ylabel("Number of Subscriber-Days")

plt.show()

### Interpretation: Distribution of Daily Cell Diversity

The distribution of daily unique cell towers per subscriber is **right-skewed**, indicating that most subscribers exhibit relatively localized daily movement, while a smaller proportion demonstrate wider geographic mobility.

Key observations:

1. **Concentration at low cell counts:** Most subscriber-days involve approximately 1–7 unique cell towers, suggesting relatively limited daily geographic movement.

2. **Peak around 1 and 6 cells:** The distribution shows noticeable peaks at these values, indicating common patterns of very localized activity and moderately broader movement.

3. **Declining frequency at higher cell counts:** The number of subscriber-days decreases as daily cell diversity increases beyond approximately 10 cells.

4. **Highly mobile subscriber-days:** A small proportion involve more than 15–20 unique cell towers, with a few reaching approximately 25 cells or more.

### Summary of Findings

Most subscribers exhibit relatively localized daily mobility, typically connecting to fewer than 8 unique cell towers per day. However, a smaller group demonstrates substantially broader geographic movement.

**Implication for the project:** Daily cell diversity can help distinguish subscribers with localized movement patterns from highly mobile subscribers. Combined with nighttime activity, repeated tower visits and temporal consistency, it provides useful features for home-location inference.

#### Approximate distance between consecutive observed cells

In [ ]:
# Sort events chronologically for each subscriber
mobility_events = df.sort_values(
    ["msisdn", "datetime"]
).copy()

# Previous coordinates for each subscriber
mobility_events["previous_latitude"] = (
    mobility_events.groupby("msisdn")["latitude"].shift(1)
)

mobility_events["previous_longitude"] = (
    mobility_events.groupby("msisdn")["longitude"].shift(1)
)

mobility_events["previous_datetime"] = (
    mobility_events.groupby("msisdn")["datetime"].shift(1)
)

# Haversine distance
def haversine_km(lat1, lon1, lat2, lon2):

    earth_radius = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * earth_radius * np.arcsin(np.sqrt(a))


mobility_events["distance_from_previous_km"] = haversine_km(
    mobility_events["previous_latitude"],
    mobility_events["previous_longitude"],
    mobility_events["latitude"],
    mobility_events["longitude"]
)

mobility_events["time_since_previous_hours"] = (
    (
        mobility_events["datetime"]
        - mobility_events["previous_datetime"]
    ).dt.total_seconds() / 3600
)

mobility_events["distance_from_previous_km"] = (
    mobility_events["distance_from_previous_km"].fillna(0)
)

display(
    mobility_events[
        [
            "msisdn",
            "datetime",
            "cell_id",
            "distance_from_previous_km",
            "time_since_previous_hours"
        ]
    ].head(20)
)

### Interpretation: Distance Between Consecutive Observed Cells

The table illustrates subscriber movement by measuring the approximate geographical distance and elapsed time between consecutive network events.

Key observations:

1. **Geographical movement:** The subscriber moves between cell towers at varying distances, ranging from 0 km to approximately 15 km in the displayed records.

2. **Repeated cell connections:** Several consecutive events have a distance of 0 km, indicating repeated connections to the same geographical location or cell coordinates.

3. **Time between events:** The time intervals vary considerably, from approximately 4 minutes to several days, suggesting irregular network activity and observation intervals.

4. **Movement patterns:** Some transitions involve relatively short distances (1–5 km), while others involve longer distances (10–15 km), potentially reflecting movement across different parts of Nairobi.

5. **First observation:** The first event for each subscriber has no previous observation, so its time difference is recorded as `NaN`. The distance is initialized to zero.

### Summary of Findings

The results demonstrate varying geographical movement patterns, with subscribers alternating between repeated connections to familiar locations and transitions across different cell towers. Both distance and elapsed time provide useful information about subscriber mobility.

**Implication for the project:** Distance between consecutive cells and time between observations can help identify movement intensity, distinguish stationary periods from potential travel, and support home-location inference. These features should be combined with repeated nighttime connections and temporal consistency to identify probable residential locations.

In [ ]:
# inspecting the distribution

valid_transitions = mobility_events[
    mobility_events["previous_datetime"].notna()
].copy()

plt.figure(figsize=(12, 6))

sns.histplot(
    data=valid_transitions[
        valid_transitions["distance_from_previous_km"] <= 50
    ],
    x="distance_from_previous_km",
    bins=50
)

plt.title("Distance Between Consecutive Observed Cell Locations")
plt.xlabel("Approximate Distance (km)")
plt.ylabel("Number of Transitions")

plt.show()

### Interpretation: Distance Between Consecutive Observed Cell Locations

The distribution is **highly right-skewed**, indicating that most consecutive network observations occur within relatively short geographical distances.

Key observations:

1. **High concentration at 0 km:** Most transitions have zero or near-zero distance, suggesting repeated connections to the same cell location or coordinates.

2. **Short-distance transitions:** A substantial number of movements occur within approximately 1–5 km, indicating relatively localized geographical movement.

3. **Declining transition frequency:** The number of transitions decreases as distance increases, with fewer observations beyond 10 km.

4. **Long-distance movements:** A small proportion of transitions exceed 15–20 km, potentially representing longer journeys or movement between distant locations.

### Summary of Findings

Most consecutive network observations involve no geographical displacement or relatively short distances. Long-distance transitions are considerably less frequent, suggesting that subscriber activity is predominantly localized.

**Implication for the project:** Consecutive cell distance is a useful mobility indicator for distinguishing localized activity from wider geographical movement. Combined with elapsed time between observations, repeated nighttime connections and cell-visit frequency, it can support home-location inference.

**Methodological consideration:** Zero-distance transitions do not necessarily indicate that subscribers were stationary; they may simply reflect repeated observations at the same tower coordinates. Distances represent straight-line displacement, not actual travel distance.

### Section 6 — Network technology and event behaviour

Examine the relationship between the two categorical variables.

#### Data type by network technology

In [ ]:
data_service_table = pd.crosstab(
    df["data_type"],
    df["service"]
)

display(data_service_table)

#### Proportions within each data type

In [ ]:
data_service_percentage = pd.crosstab(
    df["data_type"],
    df["service"],
    normalize="index"
) * 100

display(data_service_percentage.round(2))

#### Visualise the relationship

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    data_service_percentage,
    annot=True,
    fmt=".1f",
    cmap="Blues"
)

plt.title("Network Technology Distribution Within Each Data Type")
plt.xlabel("Network Technology")
plt.ylabel("Data Type")

plt.show()

### Interpretation: Network Technology Distribution Within Each Data Type

The heatmap illustrates the percentage distribution of network technologies (2G, 3G, and 4G) within CDR and IPDR records.

Key observations:

1. **CDR dominance in 2G:** Approximately 97.6% of CDR records originate from 2G networks, while only 0.8% and 1.5% originate from 3G and 4G, respectively.

2. **IPDR dominance in 4G:** Most IPDR records (52.8%) originate from 4G networks, followed by 2G (27.4%) and 3G (19.8%).

3. **Distinct technology distributions:** CDR records are overwhelmingly associated with 2G, whereas IPDR records are distributed more broadly across network technologies, particularly 4G.

### Summary of Findings

The dataset exhibits clear differences in network technology usage between CDR and IPDR records. **CDR activity is predominantly associated with 2G, while IPDR activity is primarily associated with 4G.**

**Implication for the project:** Network technology and data type should be considered during mobility analysis and home-location inference, as differences in event generation and network coverage may introduce observational biases into subscriber mobility patterns.

### Section 7 — Nighttime activity investigation

This is particularly important because the eventual home-location inference method will use recurring nighttime cell observations.

First investigate whether subscribers actually have sufficient nighttime observations to support that method.

For this initial analysis, define nighttime as 8 PM to 6 AM. This is an experimental analytical definition, not a universal definition of residence.

#### Compare daytime and nighttime activity

In [ ]:
df["period"] = np.where(
    df["is_night"],
    "Night",
    "Day"
)

period_comparison = (
    df.groupby("period")
       .agg(
           total_events=("msisdn", "size"),
           unique_subscribers=("msisdn", "nunique"),
           unique_cells=("cell_id", "nunique")
       )
)

period_comparison["events_per_subscriber"] = (
    period_comparison["total_events"] /
    period_comparison["unique_subscribers"]
)

display(period_comparison)

### Interpretation: Daytime versus Nighttime Subscriber Activity

The results show a significant difference between daytime and nighttime network activity, with substantially higher activity and geographic diversity during the day.

Key observations:

1. **Higher daytime activity:** Daytime records account for 864,823 events, compared with 207,785 nighttime events. This indicates considerably greater network activity during daytime hours.

2. **Geographic diversity:** Daytime activity spans 35,706 unique cell towers, compared with 21,294 at night, suggesting broader geographical coverage during the day.

3. **Events per subscriber:** Subscribers generate an average of 288.27 events during the day, compared with 69.47 at night.

4. **Consistent subscriber coverage:** Nearly all 3,000 subscribers appear in both periods, indicating that the difference is primarily in activity volume rather than subscriber presence.

### Summary of Findings

Daytime network activity is substantially higher and geographically more diverse than nighttime activity. Nighttime activity is comparatively lower, with subscribers connecting to fewer cell towers.

**Implication for the project:** Nighttime activity is particularly valuable for home-location inference because subscribers are more likely to remain within their residential areas during sleeping hours. Repeated nighttime connections to the same cell towers can therefore help identify probable residential locations.

#### Nighttime activity by hour

In [ ]:
night_hourly = (
    df[df["is_night"]]
    .groupby("hour")
    .size()
)

plt.figure(figsize=(12, 5))

sns.barplot(
    x=night_hourly.index,
    y=night_hourly.values
)

plt.title("Distribution of Nighttime Network Activity")
plt.xlabel("Hour")
plt.ylabel("Number of Events")

plt.show()

### Interpretation: Distribution of Nighttime Network Activity

The chart shows that nighttime network activity is concentrated primarily between **8 PM and 11 PM**, with a sharp decline during the early morning hours.

Key observations:

1. **Evening peak:** Activity is highest at 8 PM (20:00), followed by 9 PM (21:00). This suggests substantial network usage during evening hours.

2. **Late-night decline:** Activity decreases progressively from 10 PM (22:00) through 11 PM (23:00).

3. **Early morning activity:** Between midnight and 5 AM, network activity is considerably lower, reaching its lowest levels around 2–4 AM.

4. **Potential residential signal:** The reduced activity between midnight and early morning may provide a useful window for identifying subscribers' likely residential locations, particularly through repeated connections to the same cell towers.

### Summary of Findings

Nighttime network activity is heavily concentrated between 8 PM and 11 PM, while activity drops considerably between midnight and 5 AM.

**Implication for the project:** The 12 AM–5 AM window may be more suitable for home-location inference than the entire nighttime period. However, repeated overnight cell-tower observations should be used to distinguish residential locations from temporary stays and other movement patterns.

#### Nighttime cell diversity per subscriber

In [ ]:
night_events_eda = df[df["is_night"]].copy()

night_subscriber_profile = (
    night_events_eda.groupby("msisdn")
    .agg(
        nighttime_events=("cell_id", "size"),
        nighttime_cells=("cell_id", "nunique"),
        observed_nights=("date", "nunique")
    )
    .reset_index()
)

display(night_subscriber_profile.describe().T)

### Interpretation: Nighttime Cell Diversity per Subscriber

The descriptive statistics show considerable variation in subscribers' nighttime network activity and geographic movement.

**Key observations:**

1. **Nighttime events:** Subscribers generate an average of approximately 69 nighttime events, with a median of 48. This indicates that some subscribers generate considerably more nighttime activity than others.

2. **Nighttime cell diversity:** Subscribers connect to an average of 14.7 unique cell towers at night, while the median is 12. This suggests that most subscribers interact with a relatively limited number of locations overnight.

3. **Observed nights:** The average subscriber appears on approximately 20 distinct nights out of the 28-day observation period. The median is 24 nights, indicating relatively consistent nighttime presence for many subscribers.

4. **Outliers:** The maximum nighttime cell diversity is 102 towers, suggesting a small number of subscribers exhibit unusually high geographic mobility or potentially anomalous activity.

### Summary of Findings

Nighttime subscriber activity shows substantial variation, but most subscribers have relatively consistent nighttime presence, with a median of 24 observed nights and 12 unique cell towers.

**Implication for the project:** Subscribers with frequent nighttime observations and relatively low nighttime cell diversity are promising candidates for home-location inference. Repeated observations at the same cell towers can help identify likely residential locations.

#### Number of distinct nighttime cells per subscriber

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    data=night_subscriber_profile,
    x="nighttime_cells",
    bins=30
)

plt.title("Number of Unique Nighttime Cell IDs per Subscriber")
plt.xlabel("Unique Nighttime Cell IDs")
plt.ylabel("Number of Subscribers")

plt.show()

### Interpretation: Distribution of Unique Nighttime Cells per Subscriber

The histogram shows a positively skewed distribution, indicating that most subscribers connect to a relatively small number of unique cell towers during nighttime hours.

**Key observations:**

1. **Low nighttime cell diversity:** Most subscribers connect to approximately 2–20 unique cell towers at night.
2. **Right-skewed distribution:** The number of subscribers decreases as nighttime cell diversity increases.
3. **High-mobility subscribers:** A small proportion connect to more than 40 unique cells, with a few reaching approximately 80 or more.
4. **Potential outliers:** Subscribers with unusually high nighttime cell diversity may represent highly mobile individuals or unusual network behaviour.

### Summary of Findings

Most subscribers exhibit relatively limited geographic movement during nighttime hours, suggesting recurring nighttime locations.

**Implication for the project:** Subscribers with low nighttime cell diversity and frequent nighttime observations are strong candidates for home-location inference. Those with unusually high nighttime cell diversity may require additional investigation or filtering to avoid unreliable home-location assignments.

#### Nighttime observations per subscriber

In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    data=night_subscriber_profile,
    x="observed_nights",
    bins=28
)

plt.title("Number of Observed Nights per Subscriber")
plt.xlabel("Distinct Nights with at Least One Event")
plt.ylabel("Number of Subscribers")

plt.show()

### Interpretation: Nighttime Observations per Subscriber

The histogram shows the distribution of the number of distinct nights on which subscribers generated at least one network event.

**Key observations:**

1. **High nighttime presence:** A large proportion of subscribers have nighttime observations on 28 out of the 28 days, indicating consistent nighttime network activity.
2. **Variable presence:** Other subscribers have observations ranging from approximately 2 to 27 nights.
3. **Strong concentration at 28 nights:** The sharp increase at 28 nights suggests that many subscribers are consistently active throughout the observation period.
4. **Intermittent subscribers:** Subscribers with fewer observed nights may have irregular activity, temporary presence, or limited network usage.

### Summary of Findings

Most subscribers demonstrate consistent nighttime network presence throughout the 28-day observation period, although a smaller group exhibits intermittent activity.

**Implication for the project:** Subscribers with frequent nighttime observations are more reliable candidates for home-location inference because repeated nighttime activity provides stronger evidence of habitual residential locations. Subscribers with limited observations may require additional filtering or lower confidence scores.

In [ ]:
# overnight period count

night_events_eda["night_date"] = (
    night_events_eda["datetime"].dt.normalize()
    - pd.to_timedelta(
        (night_events_eda["hour"] < 6).astype(int),
        unit="D"
    )
)

night_subscriber_profile = (
    night_events_eda.groupby("msisdn")
    .agg(
        nighttime_events=("cell_id", "size"),
        nighttime_cells=("cell_id", "nunique"),
        observed_nights=("night_date", "nunique")
    )
    .reset_index()
)

#### Section 8 — Data-quality investigations

In [ ]:
# missing values

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().mean() * 100
    )
})

display(missing_summary)

In [ ]:
# Duplicate records

exact_duplicates = df.duplicated().sum()

duplicate_subscriber_datetime_cell = df.duplicated(
    subset=["msisdn", "datetime", "cell_id"]
).sum()

print("Exact duplicate rows:", exact_duplicates)

print(
    "Duplicate subscriber-datetime-cell combinations:",
    duplicate_subscriber_datetime_cell
)

In [ ]:
# Coordinate validity

invalid_latitude = ~df["latitude"].between(-90, 90)

invalid_longitude = ~df["longitude"].between(-180, 180)

invalid_coordinates = df[
    invalid_latitude | invalid_longitude
]

print("Invalid latitude records:", invalid_latitude.sum())

print("Invalid longitude records:", invalid_longitude.sum())

print("Total records with invalid coordinates:", len(invalid_coordinates))

In [ ]:
# Time gaps between subscriber events
# This is useful for understanding how regularly the subscribers appear in the records.
# It will also help determine whether a subscriber has enough observations for a reliable nighttime inference.

gap_analysis = df.sort_values(
    ["msisdn", "datetime"]
).copy()

gap_analysis["previous_datetime"] = (
    gap_analysis.groupby("msisdn")["datetime"].shift(1)
)

gap_analysis["gap_hours"] = (
    (
        gap_analysis["datetime"]
        - gap_analysis["previous_datetime"]
    ).dt.total_seconds() / 3600
)

valid_gaps = gap_analysis["gap_hours"].dropna()

display(valid_gaps.describe())

plt.figure(figsize=(12, 6))

sns.histplot(
    valid_gaps[valid_gaps <= 72],
    bins=50
)

plt.title("Distribution of Time Gaps Between Subscriber Events")
plt.xlabel("Time Gap (Hours)")
plt.ylabel("Number of Observations")

plt.show()

### Interpretation: Time Gaps Between Subscriber Events

The histogram shows a strongly right-skewed distribution, indicating that most subscribers generate consecutive network events within relatively short time intervals.

**Key observations:**

1. **Frequent network activity:** The median time gap is 0.75 hours (45 minutes), suggesting that half of consecutive subscriber events occur within approximately 45 minutes.
2. **Short observation intervals:** The 25th percentile is 0.15 hours (9 minutes), indicating frequent network observations for a substantial proportion of subscribers.
3. **Longer inactivity periods:** The 75th percentile is 1.7 hours, meaning 75% of consecutive event gaps are within approximately 1 hour and 42 minutes.
4. **Highly variable gaps:** The maximum gap is approximately 142 hours (nearly 6 days), indicating periods of prolonged subscriber inactivity.
5. **Right-skewed distribution:** Most event gaps are concentrated within the first few hours, with relatively few observations occurring after longer intervals.

### Summary of Findings

Subscriber network activity is generally frequent, with a median inter-event interval of approximately 45 minutes. However, some subscribers experience prolonged periods without network observations.

**Implication for the project:** Short inter-event intervals provide useful temporal resolution for understanding subscriber mobility. Longer gaps should be considered when inferring movement between locations, since the subscriber may have travelled between observations. Importantly, the distance between consecutive cell observations should not automatically be interpreted as the actual distance travelled.

#### Section 9 — Relationships between important variables

Investigate whether event frequency, nighttime activity and geographic diversity are related.

##### Build a subscriber-level analytical table

In [ ]:
subscriber_eda = (
    df.groupby("msisdn")
       .agg(
           total_events=("datetime", "size"),
           unique_cells=("cell_id", "nunique"),
           active_days=("date", "nunique"),
           nighttime_events=("is_night", "sum"),
           first_seen=("datetime", "min"),
           last_seen=("datetime", "max")
       )
       .reset_index()
)

subscriber_eda["nighttime_event_share"] = (
    subscriber_eda["nighttime_events"] /
    subscriber_eda["total_events"]
)

display(subscriber_eda.head())

### Interpretation: Subscriber-Level Analytical Table

The table consolidates network activity, geographic diversity, active days, and nighttime behaviour into a single subscriber-level dataset.

**Key observations:**

1. **Event frequency:** Subscriber activity varies considerably. For example, `subscribers_00001` generated 676 events, compared with only 24 for `subscribers_00000`.
2. **Geographic diversity:** Subscribers exhibit different mobility patterns, with unique cell counts ranging from 14 to 215 among the five examples.
3. **Active days:** Most subscribers were active throughout the 28-day observation period, while some had fewer active days.
4. **Nighttime activity:** Nighttime event shares vary significantly, from 11.1% to 42.6%, suggesting differences in subscribers' nighttime network usage.
5. **Observation period:** `first_seen` and `last_seen` capture each subscriber's observation window, helping distinguish consistent subscribers from those with intermittent activity.

### Summary of Findings

The subscriber-level analytical table reveals substantial differences in network activity, geographic diversity, and nighttime behaviour.

**Implication for the project:** These variables provide a foundation for investigating relationships between subscriber activity, mobility, and nighttime presence. They will also serve as useful features for subsequent home-location inference and population mobility analysis.

#### Correlation analysis

In [ ]:
correlation_columns = [
    "total_events",
    "unique_cells",
    "active_days",
    "nighttime_events",
    "nighttime_event_share"
]

correlation_matrix = subscriber_eda[
    correlation_columns
].corr(method="spearman")

plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title("Subscriber-Level Feature Correlations")
plt.show()

### Interpretation: Subscriber-Level Feature Correlations

The Spearman correlation heatmap reveals strong relationships between subscriber activity, geographic diversity, and nighttime network usage.

**Key observations:**

1. **Total events and unique cells (0.96):** Very strong positive correlation. Subscribers generating more network events generally interact with more cell towers.

2. **Total events and nighttime events (0.96):** Very strong positive correlation, indicating that highly active subscribers also tend to generate more nighttime events.

3. **Total events and active days (0.71):** Strong positive correlation. Subscribers active on more days generally generate more network events.

4. **Unique cells and nighttime events (0.90):** Very strong positive correlation, suggesting that subscribers with greater geographic diversity also generate more nighttime events.

5. **Nighttime event share:** Shows very weak correlations with total events (0.03), unique cells (-0.02), and active days (0.00). However, it has a weak positive correlation with nighttime event volume (0.29).

### Summary of Findings

Subscriber activity volume, geographic diversity, and nighttime event frequency are strongly related. However, the proportion of events occurring at night is largely independent of overall subscriber activity.

**Implications for the project:**

- Total events, unique cells, and nighttime events capture closely related aspects of subscriber activity.
- Nighttime event share provides additional behavioural information that is not explained by overall activity volume.
- For home-location inference, nighttime activity and its consistency may be more informative than simply selecting subscribers with the highest event volumes.
- Consider potential multicollinearity when selecting features for subsequent modelling.

**Overall conclusion:** Nighttime event share and nighttime activity patterns deserve particular attention because they provide behavioural signals that complement general network activity and geographic diversity.

#### Section 10 — Generate an EDA summary report

##### Dataset summary

In [ ]:
eda_summary = {
    "total_records": len(df),

    "unique_subscribers": df["msisdn"].nunique(),

    "unique_cell_ids": df["cell_id"].nunique(),

    "unique_coordinate_pairs": df[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0],

    "earliest_timestamp": df["datetime"].min(),

    "latest_timestamp": df["datetime"].max(),

    "total_days": df["date"].nunique(),

    "average_events_per_subscriber": (
        len(df) / df["msisdn"].nunique()
    ),

    "average_events_per_cell": (
        len(df) / df["cell_id"].nunique()
    ),

    "nighttime_event_percentage": (
        df["is_night"].mean() * 100
    ),

    "weekend_event_percentage": (
        df["is_weekend"].mean() * 100
    ),

    "missing_values": df.isnull().sum().sum(),

    "exact_duplicates": df.duplicated().sum(),

    "inconsistent_cell_ids": len(inconsistent_cell_ids)
}

for key, value in eda_summary.items():
    print(f"{key}: {value}")

#### Section 11 — Representativeness Assessment
The Objective is to distinguish between how much network activity we observe and whether that activity represents the wider population.

##### Subscriber observation coverage

In [ ]:
observation_coverage = (
    df.groupby("msisdn")
      .agg(
          total_events=("datetime", "size"),
          active_days=("date", "nunique"),
          observed_nights=("date", lambda x: df.loc[x.index].loc[
              df.loc[x.index, "is_night"], "date"
          ].nunique())
      )
      .reset_index()
)

observation_coverage["observation_coverage_pct"] = (
    observation_coverage["active_days"] / 28 * 100
)

observation_coverage["coverage_category"] = pd.cut(
    observation_coverage["active_days"],
    bins=[-1, 7, 14, 21, 28],
    labels=[
        "Very low (1-7 days)",
        "Low (8-14 days)",
        "Moderate (15-21 days)",
        "High (22-28 days)"
    ]
)

coverage_summary = (
    observation_coverage["coverage_category"]
    .value_counts()
    .sort_index()
    .to_frame("subscriber_count")
)

coverage_summary["subscriber_percentage"] = (
    coverage_summary["subscriber_count"]
    / len(observation_coverage) * 100
)

display(coverage_summary)

# 2. Subscriber participation over the observation period

daily_subscriber_coverage = (
    df.groupby("date")
      .agg(
          active_subscribers=("msisdn", "nunique"),
          total_events=("msisdn", "size"),
          active_cells=("cell_id", "nunique")
      )
      .reset_index()
)

daily_subscriber_coverage["subscriber_participation_pct"] = (
    daily_subscriber_coverage["active_subscribers"]
    / df["msisdn"].nunique() * 100
)

display(daily_subscriber_coverage.head())

plt.figure(figsize=(14, 5))

sns.lineplot(
    data=daily_subscriber_coverage,
    x="date",
    y="subscriber_participation_pct",
    marker="o"
)

plt.title("Daily Subscriber Participation")
plt.xlabel("Date")
plt.ylabel("Active Subscribers (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 3. Geographic activity coverage

geographic_coverage = (
    df.groupby("cell_id")
      .agg(
          total_events=("msisdn", "size"),
          unique_subscribers=("msisdn", "nunique")
      )
      .reset_index()
)

geographic_coverage_summary = pd.DataFrame({
    "metric": [
        "Unique cell IDs observed",
        "Unique subscribers observed",
        "Total network events",
        "Median events per cell",
        "Cells with fewer than 5 events",
        "Cells with fewer than 10 events"
    ],
    "value": [
        df["cell_id"].nunique(),
        df["msisdn"].nunique(),
        len(df),
        geographic_coverage["total_events"].median(),
        (geographic_coverage["total_events"] < 5).sum(),
        (geographic_coverage["total_events"] < 10).sum()
    ]
})

display(geographic_coverage_summary)


# 4. Compare subscriber share against event share

activity_concentration = (
    subscriber_eda
    .sort_values("total_events", ascending=False)
    .reset_index(drop=True)
)

activity_concentration["subscriber_rank_pct"] = (
    (activity_concentration.index + 1)
    / len(activity_concentration) * 100
)

activity_concentration["cumulative_event_share_pct"] = (
    activity_concentration["total_events"].cumsum()
    / activity_concentration["total_events"].sum() * 100
)

concentration_results = []

for group_pct in [1, 5, 10, 20, 50]:

    n = max(
        1,
        int(np.ceil(len(activity_concentration) * group_pct / 100))
    )

    top_group = activity_concentration.head(n)

    concentration_results.append({
        "top_subscriber_group_pct": group_pct,
        "subscriber_count": n,
        "share_of_all_events_pct": (
            top_group["total_events"].sum()
            / activity_concentration["total_events"].sum()
            * 100
        )
    })

concentration_summary = pd.DataFrame(concentration_results)

display(concentration_summary)

### Representativeness Assessment — Interpretation

The analysis distinguishes network activity coverage from population representativeness.

**Network activity coverage:**
- Measures how consistently subscribers and geographic locations appear in the observed network records.
- Differences in active days indicate that subscribers have different observation coverage.
- Cells with few events provide limited evidence about activity in their surrounding locations.

**Subscriber activity representation:**
- Event concentration shows whether a relatively small group of subscribers generates a disproportionate share of records.
- High event volume does not necessarily mean that a subscriber represents more people or a larger population.

**Population representativeness limitation:**
- This dataset does not provide a population sampling frame, subscriber penetration rates, or an independent population denominator.
- Therefore, we cannot establish whether the 3,000 subscribers represent the wider Kenyan population.
- We can assess the coverage and distribution of observed network activity, but cannot claim population representativeness.

**Implication for mobility intelligence:**

Residential and population inference must account for differences in observation coverage and subscriber activity. Network activity should not be treated as a direct measure of population size.

#### Section 12 — Inference Bias Assessment

The Objective is to understand how data-quality problems and subscriber behaviour could distort residential location inference.

Examine three important sources of potential bias:
1. Unequal subscriber activity.
2. Sparse subscriber observations.
3. Inconsistent cell tower coordinate

##### Quantify potential inference biases

In [ ]:
# 1. Subscriber activity imbalance

total_events = subscriber_eda["total_events"].sum()

activity_sorted = subscriber_eda.sort_values(
    "total_events",
    ascending=False
).reset_index(drop=True)

activity_sorted["event_share"] = (
    activity_sorted["total_events"] / total_events
)

activity_sorted["cumulative_event_share"] = (
    activity_sorted["event_share"].cumsum()
)

# Gini coefficient
def calculate_gini(values):

    values = np.asarray(values, dtype=float)
    values = values[values >= 0]

    if len(values) == 0 or values.sum() == 0:
        return np.nan

    values = np.sort(values)
    n = len(values)

    return (
        2 * np.sum((np.arange(1, n + 1) * values))
        / (n * values.sum())
    ) - ((n + 1) / n)


gini_activity = calculate_gini(
    subscriber_eda["total_events"]
)

print(f"Subscriber activity Gini coefficient: {gini_activity:.4f}")

print("\nShare of network events generated by the most active subscribers:")

display(concentration_summary)


# 2. Sparse observation assessment

sparse_summary = pd.DataFrame({
    "observation_threshold": [
        "7 active days or fewer",
        "14 active days or fewer",
        "21 active days or fewer",
        "Fewer than 10 nighttime events",
        "Fewer than 5 observed nights"
    ],
    "subscriber_count": [
        (subscriber_eda["active_days"] <= 7).sum(),
        (subscriber_eda["active_days"] <= 14).sum(),
        (subscriber_eda["active_days"] <= 21).sum(),
        (subscriber_eda["nighttime_events"] < 10).sum(),
        (
            observation_coverage["observed_nights"] < 5
        ).sum()
    ]
})

sparse_summary["subscriber_percentage"] = (
    sparse_summary["subscriber_count"]
    / subscriber_eda["msisdn"].nunique() * 100
)

display(sparse_summary)


# 3. Inconsistent cell coordinate assessment

inconsistent_cell_summary = pd.DataFrame({
    "metric": [
        "Total unique cell IDs",
        "Cell IDs with inconsistent coordinates",
        "Cell IDs with consistent coordinates",
        "Percentage of inconsistent cell IDs"
    ],
    "value": [
        len(cell_geo_profile),
        len(inconsistent_cell_ids),
        len(cell_geo_profile) - len(inconsistent_cell_ids),
        len(inconsistent_cell_ids) / len(cell_geo_profile) * 100
    ]
})

display(inconsistent_cell_summary)


# 4. Assess event exposure to inconsistent cell IDs

inconsistent_cell_list = inconsistent_cell_ids["cell_id"]

df["inconsistent_cell_flag"] = (
    df["cell_id"].isin(inconsistent_cell_list)
)

coordinate_exposure = pd.DataFrame({
    "metric": [
        "Events associated with inconsistent cell IDs",
        "Events associated with consistent cell IDs"
    ],
    "event_count": [
        df["inconsistent_cell_flag"].sum(),
        (~df["inconsistent_cell_flag"]).sum()
    ]
})

coordinate_exposure["event_percentage"] = (
    coordinate_exposure["event_count"] / len(df) * 100
)

display(coordinate_exposure)


# 5. Subscriber exposure to inconsistent cell IDs

subscriber_coordinate_exposure = (
    df.groupby("msisdn")
      .agg(
          total_events=("cell_id", "size"),
          inconsistent_coordinate_events=(
              "inconsistent_cell_flag", "sum"
          )
      )
      .reset_index()
)

subscriber_coordinate_exposure["inconsistent_event_share"] = (
    subscriber_coordinate_exposure["inconsistent_coordinate_events"]
    / subscriber_coordinate_exposure["total_events"]
)

print("Subscriber exposure to inconsistent cell coordinates:")

display(
    subscriber_coordinate_exposure[
        "inconsistent_event_share"
    ].describe()
)


# 6. Visualise subscriber activity imbalance

plt.figure(figsize=(12, 5))

sns.histplot(
    data=subscriber_eda,
    x="total_events",
    bins=50,
    kde=True
)

plt.title("Subscriber Activity Distribution and Potential Inference Bias")
plt.xlabel("Total Network Events")
plt.ylabel("Number of Subscribers")
plt.tight_layout()
plt.show()


# 7. Visualise inconsistent coordinate exposure

plt.figure(figsize=(8, 5))

sns.barplot(
    data=coordinate_exposure,
    x="metric",
    y="event_percentage"
)

plt.title("Network Events by Cell Coordinate Consistency")
plt.xlabel("")
plt.ylabel("Share of Network Events (%)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Inference Bias Assessment — Interpretation

Three potential sources of bias have been identified.

**1. Unequal subscriber activity**

Subscribers generate different volumes of network events. Highly active subscribers may contribute disproportionately to nighttime activity and residential location scores.

This could cause high-activity subscribers to have stronger apparent residential evidence than subscribers with fewer observations.

**Mitigation:**
- Evaluate subscribers using distinct active nights, not only event counts.
- Consider normalised or weighted residential scoring.
- Avoid allowing high event volume alone to determine residential confidence.

**2. Sparse subscriber observations**

Subscribers with fewer active days or nighttime observations provide weaker evidence of recurring residential patterns.

A subscriber observed at one location on a single night should not automatically be classified as a resident of that location.

**Mitigation:**
- Introduce minimum observation thresholds.
- Measure the number of distinct nights at each candidate location.
- Assign low-confidence or insufficient-observation classifications where evidence is weak.

**3. Inconsistent cell tower coordinates**

A cell ID associated with multiple coordinates can create false geographic movement or incorrect residential location assignments.

This could distort spatial clustering, distance calculations and home-location estimates.

**Mitigation:**
- Investigate inconsistent cell IDs before spatial inference.
- Build a validated cell-location reference table.
- Consider coordinate variation and spatial uncertainty.
- Exclude unresolved cell IDs from high-confidence inference.

**Overall implication:**

Residential location inference should account for subscriber activity imbalance, observation coverage and geographic uncertainty. These factors should influence both the inference methodology and the confidence assigned to each estimated residence.

#### Section 13 — MNO Data Limitations and Recommendations Report

In [ ]:
# Generate the limitations report
# Calculate key metrics

total_subscribers = df["msisdn"].nunique()
total_records = len(df)
total_cells = df["cell_id"].nunique()

missing_percentage = df.isnull().mean().mean() * 100

duplicate_count = df.duplicated().sum()

invalid_coordinates_count = (
    (~df["latitude"].between(-90, 90))
    | (~df["longitude"].between(-180, 180))
).sum()

inconsistent_cell_percentage = (
    len(inconsistent_cell_ids) / total_cells * 100
)

top_10_event_share = concentration_summary.loc[
    concentration_summary["top_subscriber_group_pct"] == 10,
    "share_of_all_events_pct"
].iloc[0]

subscribers_active_28_days = (
    subscriber_eda["active_days"] == 28
).sum()

subscribers_active_28_days_pct = (
    subscribers_active_28_days / total_subscribers * 100
)

nighttime_event_share = df["is_night"].mean() * 100


# Construct the report

limitations_report = pd.DataFrame([
    {
        "area": "Subscriber activity imbalance",
        "finding": (
            f"Top 10% of subscribers generate "
            f"{top_10_event_share:.2f}% of all events."
        ),
        "implication": (
            "Highly active subscribers may dominate residential "
            "location scores."
        ),
        "recommended_mitigation": (
            "Use distinct active nights, normalised scoring and "
            "subscriber-level confidence thresholds."
        ),
        "priority": "High"
    },
    {
        "area": "Observation coverage",
        "finding": (
            f"{subscribers_active_28_days_pct:.2f}% of subscribers "
            "were active on all 28 days."
        ),
        "implication": (
            "Subscribers with shorter observation windows may have "
            "insufficient evidence for reliable residence inference."
        ),
        "recommended_mitigation": (
            "Measure observation coverage and apply minimum "
            "observation requirements."
        ),
        "priority": "High"
    },
    {
        "area": "Cell coordinate consistency",
        "finding": (
            f"{inconsistent_cell_percentage:.2f}% of cell IDs have "
            "multiple recorded coordinates."
        ),
        "implication": (
            "Unresolved coordinate variation may distort geographic "
            "clustering and residential location estimates."
        ),
        "recommended_mitigation": (
            "Validate cell ID coordinates and create a trusted "
            "cell-location reference table."
        ),
        "priority": "High"
    },
    {
        "area": "Population representativeness",
        "finding": (
            "No independent population denominator or sampling frame "
            "is available."
        ),
        "implication": (
            "The observed subscribers cannot be assumed to represent "
            "the wider population."
        ),
        "recommended_mitigation": (
            "Obtain subscriber penetration information and validate "
            "aggregated estimates against independent population data."
        ),
        "priority": "High"
    },
    {
        "area": "Data completeness",
        "finding": (
            f"Overall missing-value percentage: {missing_percentage:.4f}%; "
            f"exact duplicate records: {duplicate_count:,}."
        ),
        "implication": (
            "Missing or duplicated records, if present, may affect "
            "activity counts and temporal patterns."
        ),
        "recommended_mitigation": (
            "Maintain automated completeness checks and deduplication "
            "rules."
        ),
        "priority": "Medium"
    },
    {
        "area": "Geographic coordinate validity",
        "finding": (
            f"{invalid_coordinates_count:,} records have coordinates "
            "outside valid latitude/longitude ranges."
        ),
        "implication": (
            "Invalid coordinates can produce incorrect spatial "
            "assignments and distance calculations."
        ),
        "recommended_mitigation": (
            "Validate coordinate ranges and investigate invalid "
            "records before spatial analysis."
        ),
        "priority": "High"
    },
    {
        "area": "Observation period",
        "finding": (
            f"Dataset covers {df['date'].nunique()} days."
        ),
        "implication": (
            "A 28-day window may not capture seasonal patterns, "
            "long-term residence stability or persistent displacement."
        ),
        "recommended_mitigation": (
            "Use longer observation windows where possible and "
            "evaluate sensitivity to the selected period."
        ),
        "priority": "High"
    },
    {
        "area": "Network event interpretation",
        "finding": (
            f"Nighttime events account for {nighttime_event_share:.2f}% "
            "of all records."
        ),
        "implication": (
            "A network event does not necessarily establish physical "
            "presence or residence at a location."
        ),
        "recommended_mitigation": (
            "Use recurring observations across distinct nights and "
            "communicate the distinction between network activity "
            "and physical presence."
        ),
        "priority": "High"
    }
])

display(limitations_report)


# Export the report

REPORT_PATH = PROJECT_ROOT / "reports"
REPORT_PATH.mkdir(parents=True, exist_ok=True)

limitations_report.to_csv(
    REPORT_PATH / "mno_data_limitations_report.csv",
    index=False
)

print("MNO data limitations report exported successfully.")

### MNO Data Limitations Report — Overall Assessment

The dataset provides a useful foundation for investigating subscriber mobility and residential location patterns. However, several limitations must be addressed before using the data for population-level inference.

**Key findings:**

1. Subscriber activity is unevenly distributed, meaning event volume alone may produce biased residential location estimates.
2. Observation coverage varies between subscribers, creating differences in the strength of available evidence.
3. A substantial proportion of cell IDs have multiple recorded coordinates, requiring geographic validation.
4. The dataset does not establish whether the observed subscribers represent the wider population.
5. The 28-day observation period limits conclusions about long-term residential stability and population displacement.
6. Network events are indirect observations and should not automatically be interpreted as physical presence.

**Recommended actions before residential inference:**

- Establish minimum observation and nighttime activity thresholds.
- Investigate inconsistent cell tower coordinates.
- Develop confidence scores for residential location estimates.
- Distinguish insufficient evidence from evidence of absence.
- Document data limitations and assumptions.
- Validate future population estimates against independent reference data where available.

**Overall conclusion:**

The dataset is suitable for exploratory residential mobility analysis, subject to the limitations identified above. It should not yet be treated as a validated source of population counts or confirmed residential locations.

The next stage is to develop a residential location inference methodology that explicitly accounts for these limitations.

#### Export the key summary tables

In [ ]:
REPORT_PATH = PROJECT_ROOT / "data" / "synthetic"
REPORT_PATH.mkdir(parents=True, exist_ok=True)

subscriber_eda.to_csv(
    REPORT_PATH / "subscriber_eda_summary.csv",
    index=False
)

cell_geo_profile.to_csv(
    REPORT_PATH / "cell_eda_summary.csv",
    index=False
)

daily_active_subscribers.to_csv(
    REPORT_PATH / "daily_active_subscribers.csv",
    index=False
)

print("EDA summary tables exported.")


### Exploratory Data Analysis — Findings and Interpretation

#### 1. Subscriber Behaviour

**Finding: Subscriber activity is highly variable, with a relatively small proportion of subscribers generating a disproportionate share of network events.**

The dataset contains 3,000 unique subscribers and 1,072,608 network events.

##### Key statistics

| Metric | Result |
|---|---:|
| Total subscribers | 3,000 |
| Total network events | 1,072,608 |
| Average events per subscriber | 357.54 |
| Median events per subscriber | 265 |
| 25th percentile | 95 |
| 75th percentile | 504 |
| Minimum events | 2 |
| Maximum events | 7,527 |
| Gini coefficient | 0.4912 |

##### Event concentration

| Subscriber group | Share of total events |
|---|---:|
| Top 1% | 10.48% |
| Top 5% | 21.10% |
| Top 10% | 32.54% |
| Top 20% | 50.89% |
| Top 30% | 64.96% |
| Top 50% | 83.67% |

##### Subscriber observation coverage

| Observation period | Number of subscribers |
|---|---:|
| Active all 28 days | 2,320 |
| Active fewer than 28 days | 680 |
| Active 7 days or fewer | 174 |

**Interpretation:**

Subscriber activity is not evenly distributed. The mean event count of approximately 358 is considerably higher than the median of 265, indicating a right-skewed distribution.

The top 20% of subscribers generate approximately 50.89% of all network events, while the top 10% generate 32.54%. The Gini coefficient of 0.4912 further indicates substantial inequality in event generation.

Additionally, 680 subscribers were observed on fewer than 28 days. This suggests differences in observation coverage that must be investigated before conducting subscriber-level mobility analysis.

The shorter observation periods may include tourists, but these subscribers should not be classified as tourists without verifying their simulator-generated labels.

**Conclusion:** Subscriber activity is considerably heterogeneous. Subsequent analyses should account for differences in event volume and observation coverage rather than treating every subscriber as equally observed.

---

#### 2. Geographic Behaviour

**Finding: Network activity is distributed across a large number of cell IDs, but approximately 30.67% of cell IDs have inconsistent geographic coordinates.**

### Key statistics

| Metric | Result |
|---|---:|
| Unique cell IDs | 35,956 |
| Total events | 1,072,608 |
| Average events per cell | 29.83 |
| Median events per cell | 8 |
| 75th percentile | 19 |
| Maximum events for one cell | 7,550 |
| Cell IDs with consistent coordinates | 24,930 |
| Cell IDs with multiple coordinates | 11,026 |
| Percentage with inconsistent coordinates | 30.67% |

##### Cell activity concentration

| Cell group | Share of total events |
|---|---:|
| Top 1 cell | 0.70% |
| Top 10 cells | 6.94% |
| Top 100 cells | 12.69% |
| Top 1% of cells | 20.78% |
| Top 5% of cells | 49.08% |
| Top 10% of cells | 66.22% |

**Interpretation:**

The dataset contains 35,956 unique cell IDs, with a median of only 8 events per cell. However, a small proportion of cells account for a substantial share of network activity.

More importantly, 11,026 cell IDs (30.67%) have multiple distinct latitude or longitude values. This raises a significant data-quality concern because we cannot automatically assume that every cell ID represents one fixed geographic location.

Possible explanations include identifier reuse, variations in the underlying synthetic coordinates, or other data-generation characteristics. These possibilities require further investigation.

**Conclusion:** Geographic analysis is feasible, but cell-coordinate consistency must be investigated before constructing a reliable cell lookup table or using cell IDs for home-location inference.

---

#### 3. Temporal Behaviour

**Finding: Daily subscriber participation is relatively stable throughout the observation period, while nighttime activity accounts for approximately 19.37% of all network events.**

##### Daily subscriber activity

| Metric | Result |
|---|---:|
| Observation period | 28 days |
| Average daily active subscribers | 2,775 |
| Minimum daily active subscribers | 2,735 |
| Maximum daily active subscribers | 2,794 |
| Standard deviation | 12.27 |
| Average daily subscriber participation | 92.51% |

The highest daily subscriber participation occurred on 15 February 2026, with 2,794 active subscribers. The lowest occurred on 1 February 2026, with 2,735 active subscribers.

##### Day versus night activity

Nighttime is defined as 8 PM to 6 AM.

| Metric | Result |
|---|---:|
| Total network events | 1,072,608 |
| Nighttime events | 207,785 |
| Nighttime event share | 19.37% |
| Daytime event share | 80.63% |
| Average nighttime events per subscriber | 69.26 |
| Median nighttime events per subscriber | 48 |
| 25th percentile of nighttime events | 17 |
| 75th percentile of nighttime events | 96 |

**Interpretation:**

Daily subscriber participation remains relatively stable throughout February, averaging approximately 2,775 active subscribers per day.

Network activity is predominantly daytime activity, with approximately 80.63% of observations occurring outside the defined nighttime window. Nevertheless, the 207,785 nighttime events provide a substantial observation base for investigating recurring nighttime locations.

The available summary CSV files do not contain hourly event counts. Therefore, the precise hourly activity peaks and commuting-hour patterns cannot yet be established from these summaries.

**Conclusion:** The dataset provides consistent daily subscriber participation and a substantial volume of nighttime observations. Further hourly analysis is required to understand the detailed temporal activity patterns.

---

#### 4. Home-Location Inference Feasibility

**Finding: Almost all subscribers have nighttime activity, but nighttime event volume alone is insufficient to establish reliable recurring home locations.**

#### Nighttime activity statistics

| Metric | Result |
|---|---:|
| Total subscribers | 3,000 |
| Subscribers with nighttime events | 2,991 |
| Subscribers without nighttime events | 9 |
| Average nighttime events per subscriber | 69.26 |
| Median nighttime events per subscriber | 48 |
| 25th percentile | 17 |
| 75th percentile | 96 |
| Maximum nighttime events | 1,774 |

#### Subscribers by nighttime event thresholds

| Minimum nighttime events | Subscribers |
|---|---:|
| At least 1 | 2,991 |
| At least 5 | 2,879 |
| At least 10 | 2,577 |
| At least 20 | 2,198 |
| At least 30 | 1,966 |

**Interpretation:**

Approximately 99.7% of subscribers have at least one nighttime event. Furthermore, 2,198 subscribers have at least 20 nighttime events.

However, the number of nighttime events does not necessarily indicate how many different nights a subscriber was observed.

For example, a subscriber generating 30 nighttime events over two nights provides weaker evidence of a recurring home location than a subscriber generating 30 nighttime events across 15 separate nights.

Therefore, distinct observed nights and the consistency of the dominant nighttime cell should be used to assess home-location inference feasibility.

The following experimental thresholds are proposed for subsequent investigation:

- 3 distinct nights: Minimum exploratory evidence.
- 5 distinct nights: Initial candidate threshold.
- 7 distinct nights: Stronger evidence.
- 10 distinct nights: More robust evidence.

**Conclusion:** The dataset has a substantial nighttime observation base, but the number of subscribers meeting the distinct-night thresholds must be calculated from the original event-level dataset before determining the eligible population for home-location inference.

---

#### 5. Data Quality

**Finding: Coordinate inconsistencies and unequal subscriber observation coverage are the main issues identified from the available summary files.**

##### Data-quality assessment

| Data-quality dimension | Finding |
|---|---|
| Missing subscriber summary records | No missing records indicated |
| Unique subscriber IDs | 3,000 |
| Invalid timestamps | Previously reported as 0 |
| Inconsistent cell coordinates | 11,026 cell IDs |
| Unequal subscriber observation coverage | 680 subscribers active fewer than 28 days |
| Extreme subscriber event counts | Present |
| Extreme cell event counts | Present |
| Duplicate event records | Requires event-level investigation |
| Unusual inter-event gaps | Requires event-level investigation |

#### Key observations

**1. Coordinate inconsistencies**

Approximately 30.67% of cell IDs have multiple distinct coordinate values. This must be investigated before relying on cell-level geographic analysis.

**2. Unequal subscriber observation coverage**

A total of 680 subscribers were observed on fewer than 28 days. This may reflect the synthetic tourist population, but the actual tourist labels must be checked before classification.

**3. Event concentration**

The highest-volume subscriber generated 7,527 events, while the median subscriber generated 265 events. Similarly, the most active cell generated 7,550 events compared with a median of 8 events per cell.

These differences may influence analyses based on raw event counts.

**4. Duplicate records and event gaps**

The uploaded summary files do not contain sufficient information to establish the number of duplicate events or the distribution of time gaps between consecutive subscriber observations.

These checks must be performed using the original event-level CDR dataset.

**Conclusion:** Before preprocessing, we should investigate coordinate inconsistencies, subscriber observation coverage, duplicate records and inter-event gaps. These issues could influence home-location inference and subsequent geographic estimates.

---

#### 6. Overall EDA Conclusions

The exploratory analysis reveals five important characteristics of the Nairobi synthetic mobile network dataset:

1. **Heterogeneous subscriber activity:** Network event generation is unevenly distributed, with the top 20% of subscribers accounting for approximately 50.89% of all observations.

2. **Geographic inconsistencies:** Approximately 30.67% of cell IDs have multiple distinct coordinate values, requiring investigation before spatial analysis.

3. **Stable daily participation:** An average of approximately 2,775 subscribers are active each day, representing 92.51% of the total subscriber population.

4. **Substantial nighttime observations:** Nighttime events account for approximately 19.37% of all observations, providing a useful foundation for investigating recurring nighttime locations.

5. **Data-quality considerations:** Differences in subscriber observation periods, coordinate inconsistencies and event concentration must be considered before developing the analytical pipeline.

**Overall assessment:** The dataset provides a useful foundation for developing a mobile network analytics pipeline. However, reliable residential population estimation and mobility analysis will require careful treatment of subscriber categories, observation coverage and geographic data quality.


In [ ]:

df.to_parquet(
    INTERIM_PATH / "cdr_initial.parquet",
    index=False
)

print("Initial exploration dataset saved.")
